# Test prediction and output file

1. `predict_a`: Model A offset prediction for the test rows with an NM off-block time → `work/model_a_test.parquet` (~2 min)
2. `predict`: correctors (ADS-B → OPDI → LTFM second stage) and Model B (base → ADS-B corrector → residual corrector) → `work/prediction_test.parquet` (~4 min)
3. `output`: post-processing (time shift, monthly calibration, LOBT window) and checks → `output/prediction_<N>.parquet`

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
run.predict_a()

[     20 s] [xgboost] prediction: mean offset 11.9


[     25 s] [catboost_symmetric] prediction: mean offset 16.3


[     63 s] [lightgbm_lock] prediction: mean offset 14.2


[    100 s] [lightgbm_huber] prediction: mean offset 13.9


[    122 s] [catboost_depthwise] prediction: mean offset 15.5


[    125 s] -> work/model_a_test.parquet (661,713 rows)


In [3]:
run.predict()

[    141 s] [adsb] tracked 470,846 / 661,713 rows | RMS change 98.69


[    143 s] [opdi] rows with OPDI ground events 93,061 | rule applied 13,116


[    144 s] [ltfm] applied to 87,982 rows (k = 0.3)


[    145 s] [model_b] base: 9,077 test rows without NM | mean 1501 s


[    267 s] [model_b adsb] training 712,800 rows with NM | corrected rows without NM 4,885


[    270 s] [model_b residual] training 22,455 | applied to 2,634 rows (LTFM, LFPG), k = 1.0


[    270 s] -> work/prediction_test.parquet: 670,790 rows (A 661,713, B 9,077)


In [4]:
path = run.output("notebook 04_prediction")
print(path.relative_to(config.ROOT))

[    270 s] time shift: 316,651 rows, RMS 3.53 s


[    271 s] monthly calibration: shift in 18/40 cells, RMS change 12.15


[    271 s] LOBT window: 90 rows clipped


[    271 s] monthly calibration shifts:
 apt  month  rows  published  bias  shift
EDDM      6 14372      776.6  -9.6    6.2
EDDM      7 15013      776.3  -7.6    4.1
EGLL      1 19283     1334.9 -14.4   13.0
EGLL      2 17719     1318.9  -7.7    6.3
EGLL      6 19662     1367.0 -13.8   12.4
EGLL      7 20557     1344.1 -12.8   11.4
EHAM      1 16232      982.1 -28.5   18.6
LEBL      1 12201      883.1   5.1   -0.7
LEBL      6 17067      973.3  14.3   -9.8
LEBL      7 17879      967.2  13.7   -9.3
LEMD      6 19335     1045.7 -16.2    6.9
LIRF      1 11061     1043.3 -17.2    9.3
LIRF      2 10264     1033.7 -14.7    6.8
LIRF      6 14983     1201.1  -8.2    0.3
LIRF      7 15838     1214.0  53.1  -45.3
LSZH      6 12384      728.0  13.3   -1.3
LTFM      1 22396     1088.7  55.3  -37.3
LTFM      2 19519     1045.0  44.4  -26.4


[    271 s] -> output/prediction_1.parquet | MD5 00beceb8bf7e5c9a8bc82ce112f5873f | checks ok | mean taxi 1005.8


output/prediction_1.parquet


## Checks

In [5]:
from taxi import data, output as O, postprocess as P
c = pd.read_parquet(path).rename(columns={"TAXITIME_SEC_mvt": "taxi"})
d, checks = O.check(c)
print("checks:", checks)
h = data.departures("test", ["ADEP_mvt", "MVT_TIME_UTC_mvt"])
c = c.merge(h, on="MVT_ID_mvt")
c["month"] = c.MVT_TIME_UTC_mvt.dt.month
c.pivot_table(index="ADEP_mvt", columns="month", values="taxi", aggfunc="median").round(0)

checks: {'row count': True, 'no missing': True, 'floor 60': True, 'id order': True}


month,1,2,6,7
ADEP_mvt,,,,
EDDF,891.0,843.0,859.0,851.0
EDDM,841.0,753.0,756.0,748.0
EGLL,1279.0,1288.0,1320.0,1307.0
EHAM,794.0,731.0,699.0,811.0
LEBL,828.0,870.0,937.0,929.0
LEMD,978.0,978.0,1014.0,995.0
LFPG,921.0,892.0,920.0,911.0
LIRF,978.0,965.0,1102.0,1100.0
LSZH,766.0,713.0,723.0,732.0


Airport × month means against the published monthly taxi-out average. The published value excludes invalid flights, so the raw mean can be higher.

In [6]:
y = P.published().reset_index()
o = c.groupby(["ADEP_mvt", "month"])["taxi"].agg(["mean", "median", "size"]).reset_index()
o = o.merge(y[y.YEAR == 2026][["APT_ICAO", "MONTH_NUM", "pub"]], left_on=["ADEP_mvt", "month"], right_on=["APT_ICAO", "MONTH_NUM"])
o.drop(columns=["APT_ICAO", "MONTH_NUM"]).round(1).head(40)

,ADEP_mvt,month,mean,median,size,pub
0,EDDF,1,1022.099976,891.000000,15667,853.5
1,EDDF,2,894.000000,842.700012,14654,857.2
2,EDDF,6,869.099976,859.299988,19700,867.3
3,EDDF,7,855.700012,850.900024,20650,854.2
4,EDDM,1,1007.000000,840.799988,10953,755.0
5,EDDM,2,861.099976,753.500000,10534,752.5
6,EDDM,6,771.299988,756.000000,14372,776.6
7,EDDM,7,772.099976,747.799988,15013,776.3
8,EGLL,1,1333.199951,1278.900024,19283,1334.9
9,EGLL,2,1317.599976,1288.500000,17719,1318.9
